# 🏭 ASML Equipment Data Analysis
## AI-Driven Semiconductor Manufacturing Data Processing

This notebook demonstrates the complete analysis pipeline for the SECOM semiconductor manufacturing dataset.

**Project for ASML Equipment Data Analyst / AI Internship Application**

---

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print("✅ Imports complete")

## 2. Load SECOM Dataset

In [ ]:
from src.data_loader import load_secom_data, get_data_summary, print_data_summary

# Load data
features, labels, timestamps = load_secom_data()

# Get summary
summary = get_data_summary(features, labels)
print_data_summary(summary)

In [ ]:
# Display first few rows
print("Feature Data Sample:")
display(features.head())

print(f"\nShape: {features.shape}")
print(f"Memory usage: {features.memory_usage().sum() / 1e6:.2f} MB")

## 3. Data Exploration

In [ ]:
# Class distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
label_counts = labels.value_counts()
colors = ['#2ecc71', '#e74c3c']
ax1 = axes[0]
bars = ax1.bar(['Pass (-1)', 'Fail (1)'], [label_counts[-1], label_counts[1]], color=colors)
ax1.set_title('Class Distribution', fontsize=14, fontweight='bold')
ax1.set_ylabel('Count')
for bar, count in zip(bars, [label_counts[-1], label_counts[1]]):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             f'{count}', ha='center', fontsize=12)

# Pie chart
ax2 = axes[1]
ax2.pie([label_counts[-1], label_counts[1]], labels=['Pass', 'Fail'],
        autopct='%1.1f%%', colors=colors, explode=(0, 0.1),
        shadow=True, startangle=90)
ax2.set_title('Class Proportions', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"⚠️ Class Imbalance Ratio: {label_counts[-1]/label_counts[1]:.1f}:1 (Pass:Fail)")

In [ ]:
# Missing value analysis
missing_per_feature = features.isnull().sum().sort_values(ascending=False)
missing_per_sample = features.isnull().sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Top features with missing values
ax1 = axes[0]
missing_per_feature.head(30).plot(kind='bar', ax=ax1, color='coral')
ax1.set_title('Top 30 Features by Missing Values', fontsize=12, fontweight='bold')
ax1.set_xlabel('Feature')
ax1.set_ylabel('Missing Count')
ax1.tick_params(axis='x', rotation=45)

# Distribution of missing per sample
ax2 = axes[1]
ax2.hist(missing_per_sample, bins=50, color='steelblue', edgecolor='white')
ax2.axvline(missing_per_sample.mean(), color='red', linestyle='--', 
            label=f'Mean: {missing_per_sample.mean():.0f}')
ax2.set_title('Missing Values per Sample', fontsize=12, fontweight='bold')
ax2.set_xlabel('Number of Missing Values')
ax2.set_ylabel('Frequency')
ax2.legend()

plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
from src.preprocessing import preprocess_secom_data

# Preprocess data
data = preprocess_secom_data(features, labels)

X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']
report = data['report']

print("\n📋 Preprocessing Report:")
for key, value in report.items():
    print(f"   {key}: {value}")

In [ ]:
# Visualize feature reduction
reduction_stages = {
    'Original': report['original_features'],
    'After Missing Removal': report['original_features'] - report['removed_high_missing'],
    'After Constant Removal': report['original_features'] - report['removed_high_missing'] - report['removed_constant'],
    'Final': report['final_features']
}

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(list(reduction_stages.keys()), list(reduction_stages.values()), 
               color=plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(reduction_stages))))
ax.set_xlabel('Number of Features')
ax.set_title('Feature Reduction Pipeline', fontsize=14, fontweight='bold')

for bar, val in zip(bars, reduction_stages.values()):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{val}', va='center', fontsize=11)

plt.tight_layout()
plt.show()

print(f"\n✅ Reduced from {report['original_features']} to {report['final_features']} features")
print(f"   ({100*(1-report['final_features']/report['original_features']):.1f}% reduction)")

## 5. Feature Selection

In [ ]:
from src.feature_selection import FeatureSelector

# Run feature selection
selector = FeatureSelector(n_features=50)
selector.fit(X_train, y_train)

# Get results
top_features = selector.get_top_features()
importance = selector.get_feature_importance_report()

print("\n📊 Top 15 Most Important Sensor Signals:")
print("-" * 50)
for i, feat in enumerate(top_features[:15], 1):
    avg_rank = importance[importance['feature'] == feat]['avg_rank'].values[0]
    print(f"   {i:2d}. {feat} (avg rank: {avg_rank:.1f})")

In [ ]:
# Visualize feature importance
top_20 = importance.head(20)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Combined ranking
ax1 = axes[0]
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, 20))[::-1]
ax1.barh(range(20), top_20['avg_rank'].values[::-1], color=colors)
ax1.set_yticks(range(20))
ax1.set_yticklabels(top_20['feature'].values[::-1])
ax1.set_xlabel('Average Rank (lower is better)')
ax1.set_title('Top 20 Features by Combined Ranking', fontsize=12, fontweight='bold')

# Rankings across methods
ax2 = axes[1]
rank_cols = [col for col in importance.columns if col.endswith('_rank') and 'final' not in col and 'avg' not in col]

for col in rank_cols:
    method_name = col.replace('_rank', '').replace('_', ' ').title()
    ax2.scatter(range(20), top_20[col].values, label=method_name, alpha=0.7, s=50)

ax2.set_xticks(range(20))
ax2.set_xticklabels(top_20['feature'].values, rotation=45, ha='right')
ax2.set_ylabel('Rank')
ax2.set_title('Rankings Across Methods', fontsize=12, fontweight='bold')
ax2.legend(loc='upper left', fontsize=8)
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

## 6. Model Training and Evaluation

In [ ]:
from src.models import YieldPredictor

# Select top features
X_train_selected = selector.transform(X_train)
X_test_selected = selector.transform(X_test)

print(f"Selected features: {X_train_selected.shape[1]}")

# Train models
predictor = YieldPredictor(handle_imbalance='smote')
results = predictor.train_and_evaluate(
    X_train_selected, y_train,
    X_test_selected, y_test
)

In [ ]:
# Display model comparison
print("\n📊 Model Comparison Results:")
display(results[['model_name', 'balanced_accuracy', 'sensitivity', 'specificity', 'roc_auc', 'f1_fail']].round(3))

In [ ]:
# Visualize model performance
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

models = results['model_name'].values
x = np.arange(len(models))

# Balanced Accuracy
ax1 = axes[0, 0]
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(models)))
bars = ax1.bar(x, results['balanced_accuracy'], color=colors)
ax1.set_xticks(x)
ax1.set_xticklabels(models, rotation=45, ha='right')
ax1.set_ylabel('Balanced Accuracy')
ax1.set_title('Model Comparison: Balanced Accuracy', fontweight='bold')
ax1.set_ylim(0, 1)

# Sensitivity vs Specificity
ax2 = axes[0, 1]
width = 0.35
ax2.bar(x - width/2, results['sensitivity'], width, label='Sensitivity', color='#e74c3c')
ax2.bar(x + width/2, results['specificity'], width, label='Specificity', color='#2ecc71')
ax2.set_xticks(x)
ax2.set_xticklabels(models, rotation=45, ha='right')
ax2.set_ylabel('Score')
ax2.set_title('Sensitivity vs Specificity', fontweight='bold')
ax2.legend()
ax2.set_ylim(0, 1)

# ROC AUC
ax3 = axes[1, 0]
roc_auc = results['roc_auc'].fillna(0)
ax3.bar(x, roc_auc, color='steelblue')
ax3.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Random')
ax3.set_xticks(x)
ax3.set_xticklabels(models, rotation=45, ha='right')
ax3.set_ylabel('ROC AUC')
ax3.set_title('ROC AUC Score', fontweight='bold')
ax3.legend()
ax3.set_ylim(0, 1)

# F1 Scores
ax4 = axes[1, 1]
ax4.bar(x - width/2, results['f1_fail'], width, label='F1 (Fail)', color='#e74c3c')
ax4.bar(x + width/2, results['f1_pass'], width, label='F1 (Pass)', color='#2ecc71')
ax4.set_xticks(x)
ax4.set_xticklabels(models, rotation=45, ha='right')
ax4.set_ylabel('F1 Score')
ax4.set_title('F1 Scores by Class', fontweight='bold')
ax4.legend()
ax4.set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
# Classification report
print("\n📋 Classification Report (Best Model):")
print(predictor.get_classification_report(X_test_selected, y_test))

## 7. Anomaly Detection

In [ ]:
from src.anomaly_detection import detect_equipment_anomalies

# Run anomaly detection
anomaly_results = detect_equipment_anomalies(X_train_selected, y_train)

print("\n📊 Anomaly Detection Report:")
display(anomaly_results['report'][['precision', 'recall', 'f1', 'detected_failures', 'false_alarms']])

In [ ]:
# Visualize anomaly detection
predictions = anomaly_results['predictions']
y_binary = (y_train == 1).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Anomaly score distribution
ax1 = axes[0]
pass_scores = predictions[y_binary == 0]['anomaly_score']
fail_scores = predictions[y_binary == 1]['anomaly_score']

ax1.hist(pass_scores, bins=30, alpha=0.7, label='Pass', color='#2ecc71', density=True)
ax1.hist(fail_scores, bins=30, alpha=0.7, label='Fail', color='#e74c3c', density=True)
ax1.set_xlabel('Anomaly Score')
ax1.set_ylabel('Density')
ax1.set_title('Anomaly Score Distribution by Class', fontweight='bold')
ax1.legend()

# Detection methods agreement
ax2 = axes[1]
methods = [col for col in predictions.columns if col not in ['ensemble', 'anomaly_score']]
agreement = predictions[methods].sum(axis=1)

ax2.hist(agreement[y_binary == 0], bins=range(len(methods)+2), 
         alpha=0.7, label='Pass', color='#2ecc71', align='left')
ax2.hist(agreement[y_binary == 1], bins=range(len(methods)+2), 
         alpha=0.7, label='Fail', color='#e74c3c', align='left')
ax2.set_xlabel('Number of Methods Detecting Anomaly')
ax2.set_ylabel('Count')
ax2.set_title('Detection Agreement', fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

## 8. PCA Visualization

In [ ]:
from sklearn.decomposition import PCA

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_train_selected)

fig, ax = plt.subplots(figsize=(10, 7))

scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], 
                     c=(y_train == 1).astype(int), 
                     cmap='RdYlGn_r', alpha=0.6, s=40)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PCA Visualization of SECOM Data', fontsize=14, fontweight='bold')
ax.legend(*scatter.legend_elements(), title="Class", labels=['Pass', 'Fail'])

plt.tight_layout()
plt.show()

print(f"Explained variance: PC1={pca.explained_variance_ratio_[0]*100:.1f}%, PC2={pca.explained_variance_ratio_[1]*100:.1f}%")

## 9. Summary & Conclusions

In [ ]:
print("="*70)
print("📋 ANALYSIS SUMMARY")
print("="*70)

print(f"\n🏭 Dataset:")
print(f"   - Samples: {summary['n_samples']:,}")
print(f"   - Original features: {summary['n_features']}")
print(f"   - Final features: {report['final_features']}")
print(f"   - Selected features: 50")

print(f"\n🎯 Class Imbalance:")
print(f"   - Pass: {summary['n_pass']:,} ({100-summary['fail_rate']:.1f}%)")
print(f"   - Fail: {summary['n_fail']:,} ({summary['fail_rate']:.1f}%)")
print(f"   - Ratio: {summary['imbalance_ratio']:.0f}:1")

best_result = results.iloc[0]
print(f"\n🤖 Best Model: {predictor.best_model_name_}")
print(f"   - Balanced Accuracy: {best_result['balanced_accuracy']:.3f}")
print(f"   - Sensitivity (Fail Detection): {best_result['sensitivity']:.3f}")
print(f"   - Specificity (Pass Detection): {best_result['specificity']:.3f}")

print(f"\n🔍 Top 5 Key Sensors:")
for i, feat in enumerate(top_features[:5], 1):
    print(f"   {i}. {feat}")

print("\n" + "="*70)
print("✅ Analysis Complete!")
print("="*70)